# INIT


In [ ]:
!git clone https://github.com/g-awaine/cerberus.git

Cloning into 'cerberus'...
remote: Enumerating objects: 255, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 255 (delta 0), reused 0 (delta 0), pack-reused 242 (from 1)
Receiving objects: 100% (255/255), 223.57 KiB | 9.72 MiB/s, done.
Resolving deltas: 100% (127/127), done.


In [2]:
%cd /content/cerberus

/content/cerberus


In [3]:
!wget https://warwick.ac.uk/fac/cross_fac/tia/software/cerberus/resnet34_cerberus.zip -O cerb_weights.zip
!unzip cerb_weights.zip -d ./checkpoints

--2026-08-02 07:19:46--  https://warwick.ac.uk/fac/cross_fac/tia/software/cerberus/resnet34_cerberus.zip
Resolving warwick.ac.uk (warwick.ac.uk)... 137.205.28.41
Connecting to warwick.ac.uk (warwick.ac.uk)|137.205.28.41|:443... connected.
HTTP request sent, awaiting response... 200 
Length: 115445070 (110M) [application/zip]
Saving to: ‘cerb_weights.zip’

cerb_weights.zip    100%[===================>] 110.10M  10.5MB/s    in 21s     

2026-08-02 07:20:07 (5.33 MB/s) - ‘cerb_weights.zip’ saved [115445070/115445070]

Archive:  cerb_weights.zip
   creating: ./checkpoints/resnet34_cerberus/
  inflating: ./checkpoints/resnet34_cerberus/weights.tar  
  inflating: ./checkpoints/resnet34_cerberus/settings.yml  


In [5]:
%pip install --no-cache-dir --no-build-isolation \
docopt \
joblib \
matplotlib \
numpy \
openslide-python \
pandas \
Pillow \
progress \
PyYAML \
scikit-image \
scipy \
termcolor \
tqdm \
imgaug \
opencv-python \
opencv-python-headless \
tensorboardX \
tiatoolbox \
kagglehub

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 204.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 287.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of huggingface-hub to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 239.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 235.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.4/308.4 kB 453.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 287.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 320.9 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 MB 292.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 320.7 MB/

In [6]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126


Looking in indexes: https://download.pytorch.org/whl/cu126


In [7]:
import kagglehub
import os

# Set the path environment variable
os.environ["KAGGLEHUB_CACHE"] = "./data"


# Download latest version
path = kagglehub.dataset_download("buraktaci/histopathology-intestinal-metaplasia")

print("Path to dataset files:", path)


100%|██████████| 710M/710M [00:07<00:00, 97.6MB/s] 

Extracting files...


Path to dataset files: ./data/datasets/buraktaci/histopathology-intestinal-metaplasia/versions/1


In [8]:
import torch
torch.cuda.is_available()

True

In [9]:
!mkdir -p "./data/healthy2im"
!mv "./data/datasets/buraktaci/histopathology-intestinal-metaplasia/versions/1/Control/Control" "./data/healthy2im/trainA"
!mv "./data/datasets/buraktaci/histopathology-intestinal-metaplasia/versions/1/intestinal metaplasia/intestinal metaplasia" "./data/healthy2im/trainB"


# Test



In [32]:
!python run_infer_tile.py \
    --gpu=0 \
    --batch_size=32 \
    --model="./checkpoints/resnet34_cerberus" \
    --input_dir="./data/healthy2im/trainA" \
    --output_dir="./data/healthy2im/gland_segmented_trainA" \
    --patch_input_shape=224 \
    --patch_output_shape=224

Process Patches: 100%|##########################| 13/13 [00:10<00:00,  1.20it/s]
Traceback (most recent call last):
  File "/content/cerberus/run_infer_tile.py", line 72, in <module>
    infer.process_file_list(run_args)
  File "/content/cerberus/infer/tile.py", line 415, in process_file_list
    file_path = proc_callback(proc_output, self.output_dir)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/cerberus/infer/tile.py", line 255, in proc_callback
    overlay = visualize_instances_dict_orig(src_image, pred_inst_info_dict)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/cerberus/misc/viz_utils.py", line 203, in visualize_instances_dict_orig
    inst_colour = viz_info["type_colour"][
                  ^^^^^^^^^^^^^^^^^^^^^^^^
KeyError: 2


In [3]:
import os
import yaml


GPU = "0"
BATCH_SIZE = 32
MODEL="./checkpoints/resnet34_cerberus"
INPUT_DIR="./data/healthy2im/trainA"
OUTPUT_DIR="./data/healthy2im/gland_segmented_trainA"
PATCH_INPUT_SHAPE=224
PATCH_OUTPUT_SHAPE=224
# -------------------------------------------------------------------------------------------------------

if __name__ == "__main__":

    os.environ["CUDA_VISIBLE_DEVICES"] = GPU

    input_dir = INPUT_DIR
    output_dir = OUTPUT_DIR

    run_root_dir = MODEL
    checkpoint_path = "%s/weights.tar" % run_root_dir
    with open("%s/settings.yml" % (run_root_dir)) as fptr:
        run_paramset = yaml.full_load(fptr)
    
    target_list = ['gland']

    run_args = {
        "nr_inference_workers": 2,
        "nr_post_proc_workers": 2,
        "batch_size": BATCH_SIZE,
        "input_dir": INPUT_DIR,
        "output_dir": OUTPUT_DIR,
        "patch_input_shape": PATCH_INPUT_SHAPE,
        "patch_output_shape": PATCH_OUTPUT_SHAPE,
        "patch_output_overlap": 0,
        "postproc_list": target_list,
    }

    from infer.tile import InferManager

    infer = InferManager(
        checkpoint_path=checkpoint_path,
        decoder_dict=run_paramset["dataset_kwargs"]["req_target_code"],
        model_args=run_paramset["model_kwargs"],
    )
    infer.process_file_list(run_args)

Process Patches: 100%|############################| 7/7 [00:09<00:00,  1.29s/it]
